In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
import joblib
from google.colab import drive


# 📂 Mount Google Drive
drive.mount('/content/drive')

# 📂 Load the Cleaned Laptop Dataset
csv_path = "/content/drive/MyDrive/MLOPs Project(Laptops)/EB_laptops_data_cleaned.csv"
print(f"\n📂 Loading data from: {csv_path}")
df = pd.read_csv(csv_path)

# 🧹 Drop unnecessary columns
df.drop(["Warranty"], axis=1, inplace=True)

# 🔠 Encode categorical columns (Brand, Processor, Condition)
label_cols = ["Brand", "Processor", "Condition"]
le_dict = {}
for col in label_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le  # Store encoders for later use

# Separate features and target variable
X = df.drop("Price", axis=1)
y = df["Price"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ✅ Feature processing complete.
print("✅ Feature processing complete.")
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)

# 📦 Define regression models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Support Vector Regressor": SVR(),
    "XGBoost Regressor": XGBRegressor(objective='reg:squarederror', random_state=42)
}

# Store results of model evaluation
results = []

# Evaluate each model
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    end = time.time()

    # Avoid division by zero in MAPE
    non_zero_indices = y_test != 0
    y_test_non_zero = y_test[non_zero_indices]
    y_pred_non_zero = y_pred[non_zero_indices]
    mape = np.mean(np.abs((y_test_non_zero - y_pred_non_zero) / y_test_non_zero)) * 100

    results.append({
        "Model": name,
        "R² Score": round(r2_score(y_test, y_pred), 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 2),
        "MAPE (%)": round(mape, 2),
        "Train Time (s)": round(end - start, 3)
    })

# Convert results to DataFrame and sort by MAE
results_df = pd.DataFrame(results).sort_values(by="MAE")

# Display model evaluation summary
print("\n🔍 Model Evaluation Summary:")
print(results_df.to_string(index=False))


# 🔝 Train the best model (Random Forest) again
best_model = RandomForestRegressor(random_state=42)
best_model.fit(X_train, y_train)

# Save the trained model
joblib.dump(best_model, "laptop_price_predictor.pkl")
print()
print("✅ Model saved as 'laptop_price_predictor.pkl'")
print()

# 📂 Load the trained model
model = joblib.load("laptop_price_predictor.pkl")

# 📂 Load new laptop data (without 'Price' column)
new_data_path = "/content/drive/MyDrive/MLOPs Project(Laptops)/cleaned_laptop_data.csv"
new_data = pd.read_csv(new_data_path)

# One-hot encode the new data and align columns with the training data
X_new = pd.get_dummies(new_data)
X_new = X_new.reindex(columns=X_train.columns, fill_value=0)

# Predict the prices using the trained model
predicted_prices = model.predict(X_new)

# Add predictions to the new data
new_data["Predicted Price"] = predicted_prices

# Show top rows of the result
print(new_data.head())

# 📂 Save predictions to a new CSV
new_data.to_csv("/content/drive/MyDrive/MLOPs Project(Laptops)/9_predicted_prices.csv", index=False)
print()
print("✅ Predicted prices saved to '9_predicted_prices.csv'")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📂 Loading data from: /content/drive/MyDrive/MLOPs Project(Laptops)/EB_laptops_data_cleaned.csv
✅ Feature processing complete.
Shape of X_train: (116, 6)
Shape of y_train: (116,)

🔍 Model Evaluation Summary:
                   Model  R² Score      MAE  MAPE (%)  Train Time (s)
           Random Forest    0.7499 10127.11     16.00           0.243
       XGBoost Regressor    0.6950 10440.38     16.85           0.245
           Decision Tree    0.7221 10620.43     16.59           0.005
       Linear Regression    0.6312 13429.85     20.22           0.005
        Lasso Regression    0.6312 13430.03     20.22           0.005
        Ridge Regression    0.6326 13435.82     20.23           0.005
Support Vector Regressor   -0.1071 22117.82     34.04           0.007

✅ Model saved as 'laptop_price_predictor.pkl'

    Brand              Processor  RAM(GB)  Storage(GB) 